# Generación de Lenguaje Natural con DistilGPT2
Este notebook guía paso a paso cómo utilizar un modelo preentrenado de generación de texto (DistilGPT2) para que puedas explicar cada estrategia a tus alumnos. Ideal para el curso de NLG.

## Paso 1: Instalación de dependencias
Instalamos `transformers`, que nos permite cargar modelos como DistilGPT2.

In [ ]:
!pip install transformers --quiet

## Paso 2: Cargar el modelo y el tokenizador
Usaremos el modelo `distilgpt2`, una versión liviana de GPT2 que permite generar texto rápidamente.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval();

## Paso 3: Definir un prompt
Este será el inicio de nuestro texto. Puedes cambiarlo para experimentar.

In [ ]:
prompt = "Había una vez un dragón que vivía en las montañas y"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

In [ ]:
attention_mask = torch.ones_like(input_ids)

## Paso 4: Estrategias de generación
### 1. Greedy Decoding
- El modelo elige el token más probable en cada paso.
- Muy determinista, pero puede generar texto repetitivo o corto.

In [ ]:
greedy_output = model.generate(input_ids, attention_mask=attention_mask, max_length=50, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

### 2. Beam Search
- Considera múltiples secuencias simultáneamente.
- Mejora la coherencia, pero aún es determinista.
- Beam width define cuántas secuencias se consideran.

In [ ]:
beam_output = model.generate(input_ids, attention_mask=attention_mask, max_length=50, num_beams=5, early_stopping=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

### 3. Top-k Sampling
- En lugar de elegir siempre la opción más probable, selecciona aleatoriamente entre las **k más probables**.
- Introduce diversidad.

In [ ]:
topk_output = model.generate(input_ids, attention_mask=attention_mask, do_sample=True, top_k=50, max_length=50, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(topk_output[0], skip_special_tokens=True))

### 4. Top-p (Nucleus) Sampling
- Elige de forma aleatoria dentro del **conjunto más pequeño de palabras cuya probabilidad total supera p**.
- Es más dinámico que top-k.

In [ ]:
topp_output = model.generate(input_ids, attention_mask=attention_mask, do_sample=True, top_p=0.92, top_k=0, max_length=50, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(topp_output[0], skip_special_tokens=True))

### 5. Controlando la temperatura
- `temperature < 1.0` hace al modelo más conservador.
- `temperature > 1.0` lo hace más creativo e impredecible.

In [ ]:
temperature_output = model.generate(input_ids, attention_mask=attention_mask, do_sample=True, temperature=1.5, top_k=50, max_length=50, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(temperature_output[0], skip_special_tokens=True))

## Paso 5: Comparación de resultados
Analiza con tus alumnos cómo cambia el texto generado dependiendo de la estrategia. ¿Qué diferencias notan en coherencia, fluidez, creatividad o repeticiones?

## Paso 6: Prueba con tu propio prompt
Invita a los alumnos a escribir su propio inicio de historia e intentar varias estrategias.